In [2]:
"""
MODULE 5 — Unit of Measure Consistency Checks

Checks:
- Commodities using multiple units of measure
- Number of distinct units per commodity
- Breakdown of rows per unit
- Top 20 commodities with inconsistent units
"""

import glob
import pandas as pd
from pathlib import Path

# ── Environment detection (Colab vs Local) ────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    import os as _os
    _os.listdir('/content/drive/MyDrive')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    _os.listdir('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset/Exports')
    BASE_DIR = Path('/content/drive/MyDrive/Dashboard-BR-CA-Data/Dataset')
    IN_COLAB = True
except Exception:
    BASE_DIR = Path.cwd().parent / 'Dataset'
    IN_COLAB = False

EXPORTS_DIR  = BASE_DIR / 'Exports'
NORM_FILE    = BASE_DIR / 'Normalised' / 'prices_normalised.csv'
REPORTS_DIR  = BASE_DIR / 'Reports'
LOOKER_DIR   = BASE_DIR / 'Looker'

print(f'Environment : {"Colab" if IN_COLAB else "Local"}')
print(f'Base dir    : {BASE_DIR}')
print(f'Exists      : {BASE_DIR.exists()}')


# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
# ──────────────────────────────────────────────────────────────
# LOAD (Same model as previous modules)
# ──────────────────────────────────────────────────────────────
def load_raw(filepath):
    df = pd.read_csv(filepath, skiprows=1, dtype=str)
    df.columns = df.columns.str.strip()
    df["_source_file"] = Path(filepath).name
    return df


def load_all(files):
    frames = []
    errors = []
    for f in files:
        try:
            frames.append(load_raw(f))
        except Exception as e:
            errors.append({"file": Path(f).name, "error": str(e)})
    combined = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return combined, errors
def validate_unit_measure(raw):

    df = raw.copy()

    results = {
        "summary": [],
        "examples": {}
    }

    if "Unit of measure" not in df.columns:
        results["summary"].append({
            "check": "Unit of Measure column exists",
            "count": 0,
            "pct_of_total": 0,
            "severity": "🔴 CRITICAL - COLUMN NOT FOUND"
        })
        return results

    # Clean unit column
    df["_unit"] = df["Unit of measure"].astype(str).str.strip().str.lower()

    # Distinct unit count per commodity
    unit_summary = (
        df.groupby("Commodity")["_unit"]
          .nunique()
          .reset_index(name="distinct_unit_count")
    )

    total_commodities = len(unit_summary)
    inconsistent = unit_summary[unit_summary["distinct_unit_count"] > 1]
    inconsistent_count = len(inconsistent)

    results["summary"].append({
        "check": "Commodities with multiple units",
        "count": int(inconsistent_count),
        "pct_of_total": round(inconsistent_count / total_commodities * 100, 2)
                        if total_commodities else 0,
        "severity": "🔴 CRITICAL" if inconsistent_count > 0 else "✅ OK"
    })

    # ─────────────────────────────────────────────
    # Detailed breakdown for inconsistent commodities
    # ─────────────────────────────────────────────
    if inconsistent_count > 0:

        detail = (
            df.groupby(["Commodity", "_unit"])
              .size()
              .reset_index(name="rows_per_unit")
        )

        detail = detail.merge(unit_summary, on="Commodity")

        detail = detail[detail["distinct_unit_count"] > 1]

        # Sort by worst inconsistency
        detail = detail.sort_values(
            by=["distinct_unit_count", "rows_per_unit"],
            ascending=False
        )

        results["examples"]["multi_unit_breakdown"] = detail.head(20)

    return results


# ──────────────────────────────────────────────────────────────
# SUMMARY TABLE (Top 20 Commodities by Distinct Units)
# ──────────────────────────────────────────────────────────────
# def unit_measure_table(raw):

#     df = raw.copy()

#     df["_unit"] = df.get("Unit of measure", "").astype(str).str.strip().str.lower()

#     summary = (
#         df.groupby("Commodity")
#           .agg(
#               total_rows=("Commodity", "size"),
#               distinct_units=("_unit", "nunique")
#           )
#           .reset_index()
#     )

#     summary = summary.sort_values(
#         by=["distinct_units", "total_rows"],
#         ascending=False
#     )

#     return summary.head(20)


# ──────────────────────────────────────────────────────────────
# PRINT RESULTS
# ──────────────────────────────────────────────────────────────
def print_results(results):

    print("\n" + "="*70)
    print("  MODULE 5 — UNIT OF MEASURE CONSISTENCY CHECKS")
    print("="*70 + "\n")

    for i, r in enumerate(results["summary"], 1):
        print(f"{i}. {r['check']}")
        print(f"   Count:    {r['count']:,} ({r['pct_of_total']}%)")
        print(f"   Severity: {r['severity']}\n")

    print("="*70)
    print("  EXAMPLE BREAKDOWN")
    print("="*70 + "\n")

    for key, df_example in results["examples"].items():
        print(f"Example: {key}")
        print("-"*60)
        print(df_example.to_string(index=False))
        print()


# ──────────────────────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":

    print("\nRunning Unit of Measure Consistency Checks...\n")

    export_files = glob.glob(str(EXPORTS_DIR / "**/*.csv"), recursive=True)

    if not export_files:
        print("❌ No files found.")
        exit(1)

    raw, _ = load_all(export_files)

    results = validate_unit_measure(raw)
    print_results(results)

    # table = unit_measure_table(raw)

    # print("\n" + "="*70)
    # print("TOP 20 COMMODITIES BY DISTINCT UNITS")
    # print("="*70)
    # print(table.to_string(index=False))


Running Unit of Measure Consistency Checks...


  MODULE 5 — UNIT OF MEASURE CONSISTENCY CHECKS

1. Commodities with multiple units
   Count:    9 (0.16%)
   Severity: 🔴 CRITICAL

  EXAMPLE BREAKDOWN

Example: multi_unit_breakdown
------------------------------------------------------------
                                                                                    Commodity                           _unit  rows_per_unit  distinct_unit_count
                               2844.43.19 - Radioactive elements, isotopes and compounds, nes radioactivity in gigabecquerels           1585                    2
                               2844.43.19 - Radioactive elements, isotopes and compounds, nes radioactivity in megabecquerels           1524                    2
                                                                        2804.10.00 - Hydrogen          volume in cubic metres            420                    2
                                                           